In [5]:
import os

os.chdir(r"E:\PGT\MLP\project_materials")
print(os.getcwd())

E:\PGT\MLP\project_materials


In [6]:
# %%
# # Add any additional libraries or submodules below

# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn
# Load data  
df = pd.read_csv("unicef_malawi.csv")
df.head()



,HH1,HH2,LN,FS4,CB3,CB4,CB5A,CB5B,CB7,CB11,...,HC19,TN1,WS1,WS3,WS4,WS7,WS11,WS14,WS15,HW5
0,1.0,2.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,NO,NO,PIPED WATER: PUBLIC TAP / STANDPIPE,ELSEWHERE,5.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,ELSEWHERE,YES,NO
1,1.0,3.0,1.0,1.0,5.0,YES,ECE,NaN,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITHOUT SLAB / OPEN PIT,IN OWN YARD / PLOT,NO,YES
2,1.0,4.0,2.0,2.0,16.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,6.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,YES
3,1.0,8.0,2.0,2.0,13.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO
4,1.0,10.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,NO,YES,TUBE WELL / BOREHOLE,ELSEWHERE,8.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO


In [2]:
# =========================================================
# Cell 1: 读取数据 + 删除FCF26缺失 + 划分train/test
# =========================================================

import pandas as pd
from sklearn.model_selection import train_test_split

# =========================================================
# 0. 基础设置
# =========================================================

RANDOM_STATE = 42
TEST_SIZE = 0.2

data_path = "unicef_malawi_cleaned_full.csv"

# 如果你的目标变量是 child_depression 或 y，请确认这里一致
target_col = "FCF26"

# =========================================================
# 1. 读取数据
# =========================================================

df = pd.read_csv(data_path).copy()

print("原始数据 shape:", df.shape)


# =========================================================
# 2. 删除 FCF26 缺失值
# =========================================================

df = df.dropna(subset=["FCF26"]).copy()

print("删除FCF26缺失后 shape:", df.shape)


# =========================================================
# 3. train / test split（分层抽样）
# =========================================================

train_df_full, test_df_full = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df[target_col],   # 防止类别比例变化
    random_state=RANDOM_STATE
)

print("\nTrain shape:", train_df_full.shape)
print("Test shape :", test_df_full.shape)


# =========================================================
# 4. 重置索引（推荐）
# =========================================================

train_df_full = train_df_full.reset_index(drop=True)
test_df_full = test_df_full.reset_index(drop=True)

print("\n数据划分完成 ✅")

原始数据 shape: (13162, 90)
删除FCF26缺失后 shape: (13036, 90)

Train shape: (10428, 90)
Test shape : (2608, 90)

数据划分完成 ✅


In [3]:
# =========================================================
# Cell 2: 变量筛选（可直接完整替代）
# 前提：
# - train_df_full, test_df_full 已存在
# - target_col 已存在
# =========================================================

import pandas as pd
import numpy as np

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from scipy.stats import chi2_contingency

# =========================================================
# 0. 参数设置
# =========================================================
RANDOM_STATE = 42

missing_thresh = 0.30      # 只在 train 上判断缺失率
mi_thresh = 0.001          # 数值/有序变量 MI 阈值
chi2_p_thresh = 0.05       # 分类变量卡方检验阈值

# =========================================================
# 1. 基本检查
# =========================================================
train_df = train_df_full.copy()
test_df = test_df_full.copy()

assert target_col in train_df.columns, f"{target_col} 不在 train_df 中"
assert target_col in test_df.columns, f"{target_col} 不在 test_df 中"

y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

X_train = train_df.drop(columns=[target_col]).copy()
X_test = test_df.drop(columns=[target_col]).copy()

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("target_col   :", target_col)

# =========================================================
# 2. 先按 train 缺失率做第一轮筛选
# =========================================================
missing_rate = X_train.isna().mean().sort_values(ascending=False)

keep_by_missing = missing_rate[missing_rate <= missing_thresh].index.tolist()
drop_by_missing = missing_rate[missing_rate > missing_thresh].index.tolist()

X_train = X_train[keep_by_missing].copy()
X_test = X_test[keep_by_missing].copy()

print("\n=== 缺失率筛选 ===")
print("保留变量数:", len(keep_by_missing))
print("删除变量数:", len(drop_by_missing))
if len(drop_by_missing) > 0:
    print("因缺失率过高删除的变量:")
    print(drop_by_missing)

# =========================================================
# 3. 定义变量类型
#    注意：这里把“真正有顺序”的变量单独列出来做手动映射
# =========================================================
numeric_vars_all = [
    "HH1", "HH2", "LN", "FS4",
    "CB3", "CL3", "CL13", "wscore", "WB4",
    "MA2", "LS2", "CSURV", "CDEAD"
]

# 手动定义有序变量映射
ordinal_maps = {
    # 教育水平
    "CB5A": {
        "NoSchool": 0,
        "ECE": 1,
        "PRIMARY": 2,
        "LOWER SECONDARY": 3,
        "UPPER SECONDARY": 4,
        "HIGHER": 5
    },
    "WB6A": {
        "NoSchool": 0,
        "ECE": 1,
        "PRIMARY": 2,
        "LOWER SECONDARY": 3,
        "UPPER SECONDARY": 4,
        "VOCATIONAL TRAINING": 5,
        "HIGHER": 6
    },

    # 年级
    "CB5B": {
        "0": 0,
        "CLASS/YEAR/GRADE 1": 1,
        "CLASS/YEAR/GRADE 2": 2,
        "CLASS/YEAR/GRADE 3": 3,
        "CLASS/YEAR/GRADE 4": 4,
        "CLASS/YEAR/GRADE 5": 5,
        "CLASS/GRADE 6": 6,
        "CLASS/GRADE 7": 7,
        "CLASS/GRADE 8": 8
    },
    "WB6B": {
        "0": 0,
        "CLASS/YEAR/GRADE 1": 1,
        "CLASS/YEAR/GRADE 2": 2,
        "CLASS/YEAR/GRADE 3": 3,
        "CLASS/YEAR/GRADE 4": 4,
        "CLASS/YEAR/GRADE 5": 5,
        "CLASS/GRADE 6": 6,
        "CLASS/GRADE 7": 7,
        "CLASS/GRADE 8": 8
    },

    # 阅读能力
    "WB14": {
        "CANNOT READ AT ALL": 0,
        "NO SENTENCE IN REQUIRED LANGUAGE / BRAILLE": 1,
        "ABLE TO READ ONLY PARTS OF SENTENCE": 2,
        "ABLE TO READ WHOLE SENTENCE": 3
    },

    # 功能困难程度
    "AF10": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2
    },
    "AF11": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2
    },
    "AF12": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2,
        "NO RESPONSE": np.nan
    },

    # 幸福感
    "LS1": {
        "VERY UNHAPPY": 0,
        "SOMEWHAT UNHAPPY": 1,
        "NEITHER HAPPY NOR UNHAPPY": 2,
        "SOMEWHAT HAPPY": 3,
        "VERY HAPPY": 4
    },

    # 生活变化
    "LS3": {
        "WORSENED": 0,
        "MORE OR LESS THE SAME": 1,
        "IMPROVED": 2
    },
    "LS4": {
        "WORSE": 0,
        "MORE OR LESS THE SAME": 1,
        "BETTER": 2
    },

    # 夜间安全感
    "VT20": {
        "VERY UNSAFE": 0,
        "UNSAFE": 1,
        "SAFE": 2,
        "VERY SAFE": 3,
        "NEVER WALK ALONE AFTER DARK": np.nan
    },
    "VT21": {
        "VERY UNSAFE": 0,
        "UNSAFE": 1,
        "SAFE": 2,
        "VERY SAFE": 3,
        "NEVER WALK ALONE AFTER DARK": np.nan
    },

    # 婚姻状态
    "MSTATUS": {
        "Never married/in union": 0,
        "Formerly married/in union": 1,
        "Currently married/in union": 2
    },

    # 配偶是否有其他妻子
    "MA3": {
        "NO": 0,
        "YES": 1,
        "NotApplicable": np.nan
    }
}

# 只保留当前数据中实际存在的数值变量 / 有序变量
numeric_vars = [c for c in numeric_vars_all if c in X_train.columns]
ordinal_vars = [c for c in ordinal_maps.keys() if c in X_train.columns]

# 其余都作为分类变量
categorical_vars = [
    c for c in X_train.columns
    if c not in numeric_vars + ordinal_vars
]

print("\n=== 变量类型划分 ===")
print("数值变量数:", len(numeric_vars))
print("有序变量数:", len(ordinal_vars))
print("分类变量数:", len(categorical_vars))

# =========================================================
# 4. 手动将有序变量映射为数字
# =========================================================
for col in ordinal_vars:
    mapping = ordinal_maps[col]
    X_train[col] = X_train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

# 一些本应为数值的列可能是 object，强制转数值
for col in numeric_vars:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

# =========================================================
# 5. 数值变量 + 有序变量：MI 筛选
# =========================================================
selected_num_ord = []
num_ord_vars = numeric_vars + ordinal_vars

if len(num_ord_vars) > 0:
    X_train_num_ord = X_train[num_ord_vars].copy()

    # 再保险：全部转数值
    for col in X_train_num_ord.columns:
        X_train_num_ord[col] = pd.to_numeric(X_train_num_ord[col], errors="coerce")

    # 删除全空列，避免 SimpleImputer 输出列数不一致
    all_nan_cols = X_train_num_ord.columns[X_train_num_ord.isna().all()].tolist()
    if len(all_nan_cols) > 0:
        print("\n以下 num/ord 变量在 train 中全为 NaN，已删除：")
        print(all_nan_cols)
        X_train_num_ord = X_train_num_ord.drop(columns=all_nan_cols)

    if X_train_num_ord.shape[1] > 0:
        mi_imputer = SimpleImputer(strategy="median")
        X_train_num_ord_imp = pd.DataFrame(
            mi_imputer.fit_transform(X_train_num_ord),
            columns=X_train_num_ord.columns,
            index=X_train_num_ord.index
        )

        mi_scores = mutual_info_classif(
            X_train_num_ord_imp,
            y_train.loc[X_train_num_ord_imp.index],
            random_state=RANDOM_STATE
        )

        mi_df = pd.DataFrame({
            "variable": X_train_num_ord_imp.columns,
            "mi_score": mi_scores
        }).sort_values("mi_score", ascending=False)

        selected_num_ord = mi_df.loc[
            mi_df["mi_score"] >= mi_thresh, "variable"
        ].tolist()
    else:
        mi_df = pd.DataFrame(columns=["variable", "mi_score"])
else:
    mi_df = pd.DataFrame(columns=["variable", "mi_score"])

print("\n=== 数值/有序变量 MI 结果 ===")
print(mi_df)
print("保留的数值/有序变量:", selected_num_ord)

# =========================================================
# 6. 分类变量：卡方检验筛选
# =========================================================
selected_cat = []
chi2_results = []

for col in categorical_vars:
    tmp = pd.DataFrame({
        col: X_train[col],
        target_col: y_train
    }).dropna()

    # 类别数过少或目标只有一个类别时跳过
    if tmp[col].nunique() < 2 or tmp[target_col].nunique() < 2:
        continue

    contingency = pd.crosstab(tmp[col], tmp[target_col])

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        continue

    try:
        chi2, p_value, dof, expected = chi2_contingency(contingency)
        chi2_results.append({
            "variable": col,
            "chi2_p_value": p_value
        })

        if p_value < chi2_p_thresh:
            selected_cat.append(col)

    except Exception as e:
        print(f"卡方检验跳过 {col}: {e}")

chi2_df = pd.DataFrame(chi2_results).sort_values("chi2_p_value", ascending=True)

print("\n=== 分类变量卡方检验结果 ===")
print(chi2_df)
print("保留的分类变量:", selected_cat)

# =========================================================
# 7. 合并筛选结果
# =========================================================
selected_features_first_round = selected_num_ord + selected_cat

print("\n=== 第一轮筛选后变量 ===")
print("变量总数:", len(selected_features_first_round))
print(selected_features_first_round)

# =========================================================
# 8. 生成后续可直接使用的数据
# =========================================================
train_screened = pd.concat(
    [X_train[selected_features_first_round], y_train],
    axis=1
).copy()

test_screened = pd.concat(
    [X_test[selected_features_first_round], y_test],
    axis=1
).copy()

print("\n=== 筛选后数据形状 ===")
print("train_screened shape:", train_screened.shape)
print("test_screened shape :", test_screened.shape)

# 可选：查看最终保留变量分类
selected_numeric_vars = [c for c in selected_features_first_round if c in numeric_vars]
selected_ordinal_vars = [c for c in selected_features_first_round if c in ordinal_vars]
selected_categorical_vars = [c for c in selected_features_first_round if c in categorical_vars]

print("\n=== 最终保留变量类型 ===")
print("数值变量:", selected_numeric_vars)
print("有序变量:", selected_ordinal_vars)
print("分类变量:", selected_categorical_vars)

X_train shape: (10428, 89)
X_test shape : (2608, 89)
target_col   : FCF26

=== 缺失率筛选 ===
保留变量数: 87
删除变量数: 2
因缺失率过高删除的变量:
['CL3', 'FCD5']

=== 变量类型划分 ===
数值变量数: 12
有序变量数: 15
分类变量数: 60

=== 数值/有序变量 MI 结果 ===
   variable  mi_score
0       HH1  0.031502
25  MSTATUS  0.015554
3       FS4  0.012010
22      LS4  0.010794
20      LS1  0.010110
8       MA2  0.008371
24     VT21  0.008095
23     VT20  0.007510
16     WB14  0.006472
10    CSURV  0.004869
12     CB5A  0.004414
13     WB6A  0.004275
5      CL13  0.002950
7       WB4  0.002930
1       HH2  0.002850
15     WB6B  0.001868
9       LS2  0.001809
18     AF11  0.000771
19     AF12  0.000540
11    CDEAD  0.000000
14     CB5B  0.000000
17     AF10  0.000000
21      LS3  0.000000
6    wscore  0.000000
4       CB3  0.000000
2        LN  0.000000
26      MA3  0.000000
保留的数值/有序变量: ['HH1', 'MSTATUS', 'FS4', 'LS4', 'LS1', 'MA2', 'VT21', 'VT20', 'WB14', 'CSURV', 'CB5A', 'WB6A', 'CL13', 'WB4', 'HH2', 'WB6B', 'LS2']

=== 分类变量卡方检验结果 ===
       variab

In [4]:
# =========================================================
# Cell 3: 在第一轮筛选结果基础上做手动调整
# 重点：
# - 自动筛选结果来自上一个 cell 的 selected_features_first_round
# - 手动添加变量从 train_df_full / test_df_full 中找
# - 输出 train_final / test_final，供后续模型 cell 直接使用
# =========================================================

import pandas as pd

# =========================================================
# 0. 承接前面 cell 的对象
# 前提：
# - train_df_full
# - test_df_full
# - train_df
# - test_df
# - selected_features_first_round
# - target_col
# 都已经在前面的 cell 中生成
# =========================================================

print("train_df_full shape:", train_df_full.shape)
print("test_df_full shape :", test_df_full.shape)
print("train_df shape     :", train_df.shape)
print("test_df shape      :", test_df.shape)
print("初步筛选后变量数:", len(selected_features_first_round))
print(selected_features_first_round)

# =========================================================
# 1. 安全检查
# =========================================================
print("\n=== 数据检查 ===")
print("target_col:", target_col)
print("target 是否在 train_df_full 中:", target_col in train_df_full.columns)
print("target 是否在 test_df_full 中 :", target_col in test_df_full.columns)
print("target 是否在 train_df 中     :", target_col in train_df.columns)
print("target 是否在 test_df 中      :", target_col in test_df.columns)

assert target_col in train_df_full.columns, f"{target_col} 不在 train_df_full 中"
assert target_col in test_df_full.columns, f"{target_col} 不在 test_df_full 中"

# =========================================================
# 2. 手动删除变量
# =========================================================
manual_drop_vars = [
    "VT22A", "VT22B", "VT22C", "VT22D", "VT22E", "VT22F", "VT22X"
]

# =========================================================
# 3. 手动添加变量（注意：从 train_df_full / test_df_full 里检查）
# =========================================================
theory_add_vars = [
    "CB3",
    "wscore",
    "WB5",
    "CB4"
]

# =========================================================
# 4. 强制保留变量
# =========================================================
structural_keep_vars = [
    "CL3_status",
    "CL13_status",
    "MA2_status"
]

# =========================================================
# 5. 过滤变量
#    - 删除变量：只能删掉“第一轮已选中”的变量
#    - 添加变量：必须真实存在于 train_df_full 和 test_df_full
# =========================================================
manual_drop_vars = [v for v in manual_drop_vars if v in selected_features_first_round]

theory_add_vars = [
    v for v in theory_add_vars
    if (v in train_df_full.columns)
    and (v in test_df_full.columns)
    and (v != target_col)
]

structural_keep_vars = [
    v for v in structural_keep_vars
    if (v in train_df_full.columns)
    and (v in test_df_full.columns)
    and (v != target_col)
]

manual_add_vars = sorted(set(theory_add_vars + structural_keep_vars))

# =========================================================
# 6. 生成最终变量列表
# =========================================================
selected_features_final = [
    v for v in selected_features_first_round
    if v not in manual_drop_vars
]

for v in manual_add_vars:
    if v not in selected_features_final:
        selected_features_final.append(v)

# 再做一次保险：只保留 full train/test 都存在的列
selected_features_final = [
    v for v in selected_features_final
    if (v in train_df_full.columns) and (v in test_df_full.columns) and (v != target_col)
]

print("\n手动删除变量:")
print(manual_drop_vars)

print("\n手动添加变量（从 train_df_full / test_df_full 检查后）:")
print(manual_add_vars)

print("\n最终变量数:", len(selected_features_final))
print(selected_features_final)

# =========================================================
# 7. 用 full 数据生成最终 train / test
#    注意这里用的是 train_df_full / test_df_full
#    这样手动添加的变量一定能接进去
# =========================================================
train_final = train_df_full[selected_features_final + [target_col]].copy()
test_final  = test_df_full[selected_features_final + [target_col]].copy()

print("\ntrain_final shape:", train_final.shape)
print("test_final shape :", test_final.shape)

# =========================================================
# 8. 输出变量调整记录
# =========================================================
adjustment_records = []

all_vars_seen = sorted(set(selected_features_first_round) | set(manual_add_vars))

for v in all_vars_seen:
    adjustment_records.append({
        "variable": v,
        "auto_selected": "Yes" if v in selected_features_first_round else "No",
        "manually_dropped": "Yes" if v in manual_drop_vars else "No",
        "manually_added": "Yes" if (v in manual_add_vars and v not in selected_features_first_round) else "No",
        "final_selected": "Yes" if v in selected_features_final else "No"
    })

adjustment_summary_df = pd.DataFrame(adjustment_records).sort_values(
    by=["final_selected", "auto_selected", "variable"],
    ascending=[False, False, True]
)

print("\n变量调整记录表：")
display(adjustment_summary_df)

# =========================================================
# 9. 保存输出（如果后续 cell 要直接读文件，也可以保留）
# =========================================================
adjustment_summary_df.to_csv("manual_adjustment_summary.csv", index=False)

pd.DataFrame({"selected_variable": selected_features_final}).to_csv(
    "final_selected_variables.csv", index=False
)

train_final.to_csv("train_final.csv", index=False)
test_final.to_csv("test_final.csv", index=False)

print("\n已输出：")
print("manual_adjustment_summary.csv")
print("final_selected_variables.csv")
print("train_final.csv")
print("test_final.csv")

train_df_full shape: (10428, 90)
test_df_full shape : (2608, 90)
train_df shape     : (10428, 90)
test_df shape      : (2608, 90)
初步筛选后变量数: 62
['HH1', 'MSTATUS', 'FS4', 'LS4', 'LS1', 'MA2', 'VT21', 'VT20', 'WB14', 'CSURV', 'CB5A', 'WB6A', 'CL13', 'WB4', 'HH2', 'WB6B', 'LS2', 'HW5', 'FCD2A', 'FCD2I', 'FCD2G', 'FCD2C', 'FCD2B', 'FCD2J', 'FCD2D', 'FCD2E', 'FCD2H', 'WS4', 'WS15', 'WS14', 'DV1B', 'VT22C', 'VT22B', 'VT1', 'DV1C', 'VT9', 'VT22A', 'TA14', 'VT22E', 'VT22D', 'VT22X', 'VT22F', 'disability', 'WB5', 'HC19', 'HC11', 'HC13', 'HC8', 'HC12', 'WS11', 'HC14', 'MA2_status', 'CL3_status', 'WS1', 'WS7', 'HC5', 'ethnicity', 'HH7', 'HC4', 'CL2', 'CL12', 'CL13_status']

=== 数据检查 ===
target_col: FCF26
target 是否在 train_df_full 中: True
target 是否在 test_df_full 中 : True
target 是否在 train_df 中     : True
target 是否在 test_df 中      : True

手动删除变量:
['VT22A', 'VT22B', 'VT22C', 'VT22D', 'VT22E', 'VT22F', 'VT22X']

手动添加变量（从 train_df_full / test_df_full 检查后）:
['CB3', 'CB4', 'CL13_status', 'CL3_status', 'MA2

,variable,auto_selected,manually_dropped,manually_added,final_selected
2,CB5A,Yes,No,No,Yes
3,CL12,Yes,No,No,Yes
4,CL13,Yes,No,No,Yes
5,CL13_status,Yes,No,No,Yes
6,CL2,Yes,No,No,Yes
...,...,...,...,...,...
45,VT22C,Yes,Yes,No,No
46,VT22D,Yes,Yes,No,No
47,VT22E,Yes,Yes,No,No
48,VT22F,Yes,Yes,No,No



已输出：
manual_adjustment_summary.csv
final_selected_variables.csv
train_final.csv
test_final.csv


In [6]:
train_final.isna().sum().sort_values(ascending=False).head(20)

HW5      2753
MA2      2489
WB14     2292
CL13     1573
FCD2A    1479
FCD2I    1454
FCD2G    1449
FCD2B    1448
FCD2C    1448
FCD2J    1446
FCD2D    1445
FCD2E    1445
FCD2H    1444
WS4      1402
WS15      762
WS14      762
LS4       741
LS2       276
DV1B      268
VT1       254
dtype: int64

In [5]:
#cell4
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

# =========================================================
# 0. 读入数据
# =========================================================

train_df = pd.read_csv("train_final.csv")
test_df = pd.read_csv("test_final.csv")

target_col = "FCF26"
random_state = 42

# 使用手动筛选后的变量
baseline_vars = selected_features_final

X_train = train_df[baseline_vars].copy()
y_train = train_df[target_col].copy()

X_test = test_df[baseline_vars].copy()
y_test = test_df[target_col].copy()
# =========================================================
# 0.1 目标变量编码：No -> 0, YES -> 1
# =========================================================
label_map = {
    "No": 0,
    "YES": 1
}

y_train = train_df[target_col].map(label_map)
y_test = test_df[target_col].map(label_map)
print("进入 baseline 的变量数:", len(baseline_vars))
print(baseline_vars)


# =========================================================
# 1. 识别变量类型
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]


# =========================================================
# 2. 预处理 pipeline
# =========================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])


# =========================================================
# 3. Baseline Logistic Regression
# =========================================================

baseline_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
    random_state=random_state
)

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", baseline_model)
])

baseline_pipeline.fit(X_train, y_train)


# =========================================================
# 4. 评估函数
# =========================================================

def evaluate_model(model, X, y):

    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    return {
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }, y_pred, y_prob


baseline_train_metrics, baseline_train_pred, baseline_train_prob = evaluate_model(
    baseline_pipeline, X_train, y_train
)

baseline_test_metrics, baseline_test_pred, baseline_test_prob = evaluate_model(
    baseline_pipeline, X_test, y_test
)


baseline_perf_df = pd.DataFrame(
    [baseline_train_metrics, baseline_test_metrics],
    index=["train", "test"]
)

print("\nBaseline表现：")
display(baseline_perf_df)


# =========================================================
# 5. 提取系数
# =========================================================

feature_names_after_encoding = baseline_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

baseline_coef_df = pd.DataFrame({
    "feature": feature_names_after_encoding,
    "coef": baseline_pipeline.named_steps["model"].coef_[0]
}).sort_values("coef", key=np.abs, ascending=False)

display(baseline_coef_df.head(20))


# =========================================================
# 6. 预测结果（供后续cell使用）
# =========================================================

baseline_train_pred_df = pd.DataFrame({
    "y_true": y_train,
    "y_pred": baseline_train_pred,
    "y_prob": baseline_train_prob
})

baseline_test_pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": baseline_test_pred,
    "y_prob": baseline_test_prob
})

进入 baseline 的变量数: 58
['HH1', 'MSTATUS', 'FS4', 'LS4', 'LS1', 'MA2', 'VT21', 'VT20', 'WB14', 'CSURV', 'CB5A', 'WB6A', 'CL13', 'WB4', 'HH2', 'WB6B', 'LS2', 'HW5', 'FCD2A', 'FCD2I', 'FCD2G', 'FCD2C', 'FCD2B', 'FCD2J', 'FCD2D', 'FCD2E', 'FCD2H', 'WS4', 'WS15', 'WS14', 'DV1B', 'VT1', 'DV1C', 'VT9', 'TA14', 'disability', 'WB5', 'HC19', 'HC11', 'HC13', 'HC8', 'HC12', 'WS11', 'HC14', 'MA2_status', 'CL3_status', 'WS1', 'WS7', 'HC5', 'ethnicity', 'HH7', 'HC4', 'CL2', 'CL12', 'CL13_status', 'CB3', 'CB4', 'wscore']

Baseline表现：


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
train,0.628980,0.617409,0.638742,0.740533,0.685881,0.669074
test,0.607362,0.594860,0.620442,0.727400,0.669677,0.633239


,feature,coef
134,cat__WS4_13.0,-7.169976
169,cat__WS4_34.0,-5.724564
83,cat__CL13_45.0,5.330232
152,cat__WS4_21.0,-4.880052
284,cat__HC5_ROOFING SHINGLES,-4.773868
136,cat__WS4_135.0,4.571785
206,cat__WS15_NO RESPONSE,4.110715
189,cat__WS4_610.0,-3.652375
270,cat__WS1_TANKER-TRUCK,-3.503921
180,cat__WS4_46.0,-3.451678


In [9]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

# =========================================================
# 0. 读入筛选后的 train / test
# =========================================================

train_df = pd.read_csv("train_final.csv")
test_df = pd.read_csv("test_final.csv")


target_col = "FCF26"
random_state = 42
Cs_grid = np.logspace(-2, 2, 30)


# 使用手动筛选变量
final_vars = selected_features_final

X_train = train_df[final_vars].copy()
y_train = train_df[target_col].map({"No": 0, "YES": 1}).copy()

X_test = test_df[final_vars].copy()
y_test = test_df[target_col].map({"No": 0, "YES": 1}).copy()

print("进入 Final(L1) 的变量数:", len(final_vars))


# =========================================================
# 1. 识别变量类型
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]


# =========================================================
# 2. 预处理 pipeline
# =========================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])


# =========================================================
# 3. Final：L1 LogisticRegressionCV
# =========================================================

final_model = LogisticRegressionCV(
    Cs=Cs_grid,
    cv=5,
    penalty="l1",
    solver="saga",
    scoring="roc_auc",
    max_iter=8000,
    random_state=random_state,
    n_jobs=-1,
    refit=True
)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model)
])

final_pipeline.fit(X_train, y_train)

best_C_final = final_pipeline.named_steps["model"].C_[0]
print("\nFinal(L1)最优 C:", best_C_final)


# =========================================================
# 4. 评估函数
# =========================================================

def evaluate_model(model, X, y):

    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    return {
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }, y_pred, y_prob


final_train_metrics, final_train_pred, final_train_prob = evaluate_model(
    final_pipeline, X_train, y_train
)

final_test_metrics, final_test_pred, final_test_prob = evaluate_model(
    final_pipeline, X_test, y_test
)


final_perf_df = pd.DataFrame(
    [final_train_metrics, final_test_metrics],
    index=["train", "test"]
)

print("\nFinal(L1)表现：")
display(final_perf_df)


# =========================================================
# 5. 提取 L1 非零变量
# =========================================================

feature_names_after_encoding = final_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

coefs = final_pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature_after_encoding": feature_names_after_encoding,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs)
})

selected_coef_df = coef_df[coef_df["coefficient"] != 0].copy()

selected_coef_df = selected_coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)


# 还原到原始变量层面

original_feature_names = X_train.columns.tolist()


def recover_original_variable(encoded_name):

    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "")

    elif encoded_name.startswith("cat__"):

        temp = encoded_name.replace("cat__", "")

        sorted_cols = sorted(
            original_feature_names,
            key=len,
            reverse=True
        )

        for col in sorted_cols:

            if temp == col or temp.startswith(col + "_"):
                return col

        return temp

    else:
        return encoded_name


selected_coef_df["original_variable"] = selected_coef_df[
    "feature_after_encoding"
].apply(recover_original_variable)


final_original_vars = selected_coef_df[
    "original_variable"
].drop_duplicates().tolist()


print("\nL1筛选后保留下来的变量数:", len(final_original_vars))

display(final_original_vars)


# =========================================================
# 6. 预测结果（供后续 cell 使用）
# =========================================================

final_train_pred_df = pd.DataFrame({
    "y_true": y_train,
    "y_pred": final_train_pred,
    "y_prob": final_train_prob
})

final_test_pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": final_test_pred,
    "y_prob": final_test_prob
})


进入 Final(L1) 的变量数: 58

Final(L1)最优 C: 0.09236708571873861

Final(L1)表现：


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
train,0.621020,0.607515,0.628483,0.751227,0.684395,0.652254
test,0.613113,0.599094,0.621795,0.747722,0.678969,0.639598



L1筛选后保留下来的变量数: 49


['WS7',
 'ethnicity',
 'VT20',
 'WS4',
 'WS14',
 'VT1',
 'LS1',
 'HH7',
 'FCD2H',
 'VT21',
 'WS11',
 'MSTATUS',
 'LS2',
 'HC8',
 'HW5',
 'wscore',
 'LS4',
 'WS1',
 'FCD2E',
 'disability',
 'CL3_status',
 'CL2',
 'MA2_status',
 'TA14',
 'MA2',
 'WS15',
 'FCD2B',
 'DV1B',
 'FCD2D',
 'FCD2A',
 'CL12',
 'CL13_status',
 'CL13',
 'WB6B',
 'WB14',
 'CB3',
 'HC12',
 'HH1',
 'WB6A',
 'CB5A',
 'VT9',
 'FCD2C',
 'HC4',
 'FCD2J',
 'HC13',
 'HH2',
 'FS4',
 'DV1C',
 'WB4']

In [63]:
# =========================================================
# Random Forest on L1-selected variables + hyperparameter tuning
# =========================================================

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

RANDOM_STATE = 42

# =========================================================
# 0. 基础检查
# =========================================================

print("target_col =", target_col)

assert target_col in train_df.columns, f"{target_col} 不在 train_df 中"
assert target_col in test_df.columns, f"{target_col} 不在 test_df 中"

# 如果你当前 L1 筛选后的变量名列表不是 l1_selected_vars，请在这里替换
selected_vars = final_original_vars.copy()

# 只保留 train/test 中都实际存在的列
selected_vars = [
    c for c in selected_vars
    if c in train_df.columns and c in test_df.columns and c != target_col
]

print("L1筛选后可用于RF的变量数:", len(selected_vars))
print(selected_vars)

# =========================================================
# 1. 构造 X / y
# =========================================================

X_train = train_df[selected_vars].copy()
X_test  = test_df[selected_vars].copy()

y_train = train_df[target_col].copy()
y_test  = test_df[target_col].copy()

print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

# 如果 y 不是 0/1，可按需要取消注释进行转换
# y_train = y_train.astype(int)
# y_test = y_test.astype(int)

# =========================================================
# 2. 自动识别变量类型
# =========================================================

numeric_vars = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_vars = [c for c in X_train.columns if c not in numeric_vars]

print("\n数值变量数:", len(numeric_vars))
print("分类变量数:", len(categorical_vars))

# =========================================================
# 3. 预处理
#   - 数值变量：中位数插补
#   - 分类变量：众数插补 + One-hot
#   注意：全部放进 pipeline，避免数据泄露
# =========================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_vars),
        ("cat", categorical_transformer, categorical_vars)
    ],
    remainder="drop"
)

# =========================================================
# 4. 建立随机森林 pipeline
# =========================================================

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# =========================================================
# 5. 调参范围
#   说明：
#   - 如果数据类别不平衡，可以保留 class_weight='balanced'
#   - n_estimators 不宜太小
# =========================================================

param_dist = {
    "classifier__n_estimators": [200, 300, 500, 800],
    "classifier__max_depth": [None, 5, 10, 15, 20, 30],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "classifier__bootstrap": [True],
    "classifier__class_weight": [None, "balanced"]
}

# =========================================================
# 6. 交叉验证 + 随机搜索
#   scoring:
#   - 若你更关注少数类识别，可用 "f1"
#   - 若类别不平衡比较明显，也可用 "balanced_accuracy"
#   - 若更关注概率排序，可用 "roc_auc"
# =========================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=40,                 # 可调大到 60 / 80
    scoring="f1",              # 可改成 "balanced_accuracy" 或 "roc_auc"
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True
)

# =========================================================
# 7. 拟合
# =========================================================

rf_search.fit(X_train, y_train)

print("\n================ Best CV Result ================")
print("Best CV score:", rf_search.best_score_)
print("Best params:")
for k, v in rf_search.best_params_.items():
    print(f"{k}: {v}")

best_rf_model = rf_search.best_estimator_

# =========================================================
# 8. 在训练集和测试集上评估
# =========================================================

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# ---------- train ----------
y_train_pred = best_rf_model.predict(X_train)
y_train_proba = best_rf_model.predict_proba(X_train)[:, 1]

# ---------- test ----------
y_test_pred = best_rf_model.predict(X_test)
y_test_proba = best_rf_model.predict_proba(X_test)[:, 1]

# ---------- 汇总函数 ----------
def get_metrics(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba)
    }

train_metrics = get_metrics(y_train, y_train_pred, y_train_proba)
test_metrics = get_metrics(y_test, y_test_pred, y_test_proba)

# ---------- 打印结果 ----------
print("\n================ Training Set Performance ================")
for k, v in train_metrics.items():
    print(f"{k:<18}: {v:.4f}")

print("\nConfusion Matrix (Train):")
print(confusion_matrix(y_train, y_train_pred))

print("\nClassification Report (Train):")
print(classification_report(y_train, y_train_pred, digits=4, zero_division=0))

print("\n================ Test Set Performance ================")
for k, v in test_metrics.items():
    print(f"{k:<18}: {v:.4f}")

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))

# =========================================================
# 9. 提取特征重要性
#   注意：经过 one-hot 后，特征名会展开
# =========================================================

feature_names_after_preprocess = best_rf_model.named_steps["preprocessor"].get_feature_names_out()
importances = best_rf_model.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names_after_preprocess,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\n================ Top 30 Feature Importances ================")
print(importance_df.head(30))

# 如需保存：
# importance_df.to_csv("rf_feature_importance_from_l1_selected_vars.csv", index=False)

target_col = FCF26
L1筛选后可用于RF的变量数: 50
['WS7', 'ethnicity', 'VT20', 'WS4', 'WS14', 'VT1', 'LS1', 'HH7', 'FCD2H', 'VT21', 'WS11', 'MSTATUS', 'LS2', 'HC8', 'HW5', 'wscore', 'LS4', 'WS1', 'FCD2E', 'disability', 'CL2', 'CL3_status', 'MA2_status', 'TA14', 'MA2', 'WS15', 'FCD2B', 'DV1B', 'CL13_status', 'CL12', 'FCD2D', 'FCD2A', 'WB6B', 'CL13', 'WB14', 'CB3', 'HC12', 'HH1', 'WB6A', 'VT9', 'CB5A', 'FCD2C', 'HC4', 'FCD2J', 'CB7', 'HC13', 'HH2', 'FS4', 'DV1C', 'WB4']

X_train shape: (10428, 50)
X_test shape : (2608, 50)
y_train shape: (10428,)
y_test shape : (2608,)

数值变量数: 8
分类变量数: 42
Fitting 5 folds for each of 40 candidates, totalling 200 fits


KeyboardInterrupt: 

In [10]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

# =========================================================
# 0. 读入数据
# =========================================================

train_df = pd.read_csv("train_final.csv")
test_df = pd.read_csv("test_final.csv")

target_col = "FCF26"
random_state = 42

# =========================================================
# 1. 使用 L1 筛选后的变量
# 优先使用 final_original_vars
# =========================================================

try:
    final_vars_xgb = final_original_vars.copy()
    print("使用 L1 非零系数筛选后的变量。")
except NameError:
    final_vars_xgb = selected_features_final.copy()
    print("未检测到 final_original_vars，退回使用 selected_features_final。")

print("进入 XGBoost final model 的变量数:", len(final_vars_xgb))
print(final_vars_xgb)

# =========================================================
# 2. 准备 X / y
# 把 y 统一编码成 0/1
# =========================================================

X_train = train_df[final_vars_xgb].copy()
X_test = test_df[final_vars_xgb].copy()

y_train = (
    train_df[target_col]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"NO": 0, "YES": 1})
    .copy()
)

y_test = (
    test_df[target_col]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"NO": 0, "YES": 1})
    .copy()
)

print("\ny_train unique:", y_train.unique())
print("y_test unique :", y_test.unique())
print("y_train missing:", y_train.isna().sum())
print("y_test missing :", y_test.isna().sum())

# 如有标签映射失败，直接报错
if y_train.isna().sum() > 0 or y_test.isna().sum() > 0:
    raise ValueError("目标变量 FCF26 中存在无法映射为 0/1 的取值，请先检查原始标签。")

# =========================================================
# 3. 识别变量类型
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]

print("\n数值变量数:", len(numeric_cols))
print("分类变量数:", len(categorical_cols))

# =========================================================
# 4. 预处理
# XGBoost 对树模型不需要标准化，但保留数值插补
# 分类变量仍需 one-hot
# =========================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# =========================================================
# 5. 计算类别不平衡权重
# scale_pos_weight = negatives / positives
# =========================================================

n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

if n_pos == 0:
    raise ValueError("训练集中正类数量为 0，无法训练二分类模型。")

scale_pos_weight_value = n_neg / n_pos
print("\nscale_pos_weight:", scale_pos_weight_value)

# =========================================================
# 6. 建立 pipeline
# =========================================================

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=-1,
        tree_method="hist",
        scale_pos_weight=scale_pos_weight_value
    ))
])

# =========================================================
# 7. 参数网格
# 这是一个相对稳健、规模适中的调参范围
# =========================================================

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 4, 5],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
    "classifier__min_child_weight": [1, 3, 5],
    "classifier__gamma": [0, 0.1, 0.3]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

grid_search_xgb = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

# =========================================================
# 8. 拟合
# =========================================================

grid_search_xgb.fit(X_train, y_train)

best_xgb_model = grid_search_xgb.best_estimator_

print("\n=== Best XGBoost Parameters ===")
print(grid_search_xgb.best_params_)

print("\n=== Best CV ROC-AUC ===")
print(grid_search_xgb.best_score_)

# =========================================================
# 9. 评估函数
# =========================================================

def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    metrics_dict = {
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }

    return metrics_dict, y_pred, y_prob

xgb_train_metrics, xgb_train_pred, xgb_train_prob = evaluate_model(
    best_xgb_model, X_train, y_train
)

xgb_test_metrics, xgb_test_pred, xgb_test_prob = evaluate_model(
    best_xgb_model, X_test, y_test
)

# =========================================================
# 10. 输出结果
# =========================================================

xgb_perf_df = pd.DataFrame(
    [xgb_train_metrics, xgb_test_metrics],
    index=["train", "test"]
)

print("\n=== XGBoost Final Model Performance ===")
display(xgb_perf_df)

print("\n=== Test Confusion Matrix ===")
print(confusion_matrix(y_test, xgb_test_pred))

print("\n=== Test Classification Report ===")
print(classification_report(y_test, xgb_test_pred, digits=4))

# =========================================================
# 11. 保存预测结果，供后续 cell 使用
# =========================================================

xgb_train_pred_df = pd.DataFrame({
    "y_true": y_train,
    "y_pred": xgb_train_pred,
    "y_prob": xgb_train_prob
})

xgb_test_pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": xgb_test_pred,
    "y_prob": xgb_test_prob
})

# =========================================================
# 12. 提取特征重要性（基于 one-hot 后特征）
# =========================================================

feature_names_after_encoding = best_xgb_model.named_steps[
    "preprocessor"
].get_feature_names_out()

xgb_importances = best_xgb_model.named_steps[
    "classifier"
].feature_importances_

xgb_importance_df = pd.DataFrame({
    "feature_after_encoding": feature_names_after_encoding,
    "importance": xgb_importances
}).sort_values("importance", ascending=False)

print("\n=== Top 30 Encoded Feature Importances ===")
display(xgb_importance_df.head(30))

# =========================================================
# 13. 如需还原到原始变量层面
# =========================================================

original_feature_names = X_train.columns.tolist()

def recover_original_variable(encoded_name):
    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "")
    elif encoded_name.startswith("cat__"):
        temp = encoded_name.replace("cat__", "")
        sorted_cols = sorted(original_feature_names, key=len, reverse=True)
        for col in sorted_cols:
            if temp == col or temp.startswith(col + "_"):
                return col
        return temp
    else:
        return encoded_name

xgb_importance_df["original_variable"] = xgb_importance_df[
    "feature_after_encoding"
].apply(recover_original_variable)

xgb_original_importance_df = (
    xgb_importance_df
    .groupby("original_variable", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

print("\n=== Top Original Variables by Aggregated Importance ===")
display(xgb_original_importance_df.head(20))

使用 L1 非零系数筛选后的变量。
进入 XGBoost final model 的变量数: 49
['WS7', 'ethnicity', 'VT20', 'WS4', 'WS14', 'VT1', 'LS1', 'HH7', 'FCD2H', 'VT21', 'WS11', 'MSTATUS', 'LS2', 'HC8', 'HW5', 'wscore', 'LS4', 'WS1', 'FCD2E', 'disability', 'CL3_status', 'CL2', 'MA2_status', 'TA14', 'MA2', 'WS15', 'FCD2B', 'DV1B', 'FCD2D', 'FCD2A', 'CL12', 'CL13_status', 'CL13', 'WB6B', 'WB14', 'CB3', 'HC12', 'HH1', 'WB6A', 'CB5A', 'VT9', 'FCD2C', 'HC4', 'FCD2J', 'HC13', 'HH2', 'FS4', 'DV1C', 'WB4']

y_train unique: [0 1]
y_test unique : [1 0]
y_train missing: 0
y_test missing : 0

数值变量数: 8
分类变量数: 41

scale_pos_weight: 0.8281907433380085
Fitting 5 folds for each of 972 candidates, totalling 4860 fits

=== Best XGBoost Parameters ===
{'classifier__colsample_bytree': 0.8, 'classifier__gamma': 0, 'classifier__learning_rate': 0.03, 'classifier__max_depth': 5, 'classifier__min_child_weight': 5, 'classifier__n_estimators': 300, 'classifier__subsample': 0.8}

=== Best CV ROC-AUC ===
0.6620348477580735

=== XGBoost Final Model Perf

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
train,0.741273,0.741187,0.775275,0.742111,0.758330,0.821258
test,0.615031,0.612597,0.651179,0.638402,0.644728,0.655025



=== Test Confusion Matrix ===
[[693 488]
 [516 911]]

=== Test Classification Report ===
              precision    recall  f1-score   support

           0     0.5732    0.5868    0.5799      1181
           1     0.6512    0.6384    0.6447      1427

    accuracy                         0.6150      2608
   macro avg     0.6122    0.6126    0.6123      2608
weighted avg     0.6159    0.6150    0.6154      2608


=== Top 30 Encoded Feature Importances ===


,feature_after_encoding,importance
114,cat__LS1_VERY HAPPY,0.015329
170,cat__CL3_status_structural_missing,0.013151
124,cat__VT21_VERY SAFE,0.012052
24,cat__VT20_VERY SAFE,0.011742
110,cat__VT1_YES,0.011295
106,cat__WS14_IN OWN DWELLING,0.011224
109,cat__VT1_NO,0.010194
175,cat__MA2_status_structural_missing,0.009963
11,"cat__WS7_YES, AT LEAST ONCE",0.009771
171,cat__CL2_False,0.009191



=== Top Original Variables by Aggregated Importance ===


,original_variable,importance
44,WS4,0.085037
3,CL13,0.073193
47,ethnicity,0.058885
40,WS1,0.055743
39,WB6B,0.048226
25,LS1,0.041315
34,VT21,0.037202
33,VT20,0.032877
41,WS11,0.030950
38,WB6A,0.024695
